# Edge Bandwidth and Privacy Analysis

This notebook estimates bandwidth savings from edge-only PPE violation logging compared with continuous raw video streaming. It supports actual attached video files and falls back to reproducible bitrate assumptions.

## 1. Controls

Use actual video sizes when available. Otherwise, the fallback assumes a standard compressed 1080p stream.

In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
RESULTS_DIR = KAGGLE_WORKING / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FAST_DEBUG = True
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
ASSUMED_BITRATE_MBPS = 5.0
ASSUMED_DURATION_SECONDS = 10 * 60
ASSUMED_ALERTS_PER_HOUR = 60
LOG_BYTES_PER_ALERT = 512
SNAPSHOT_KB_PER_ALERT = 120
BLURRED_SNAPSHOT_KB_PER_ALERT = 80
MAX_VIDEO_FILES = 5 if FAST_DEBUG else 50

print('ASSUMED_BITRATE_MBPS:', ASSUMED_BITRATE_MBPS)
print('ASSUMED_DURATION_SECONDS:', ASSUMED_DURATION_SECONDS)

## 2. Environment and Video Discovery

This checks the runtime and searches Kaggle inputs for representative video files.

In [ ]:
import json
import subprocess
import sys

import numpy as np
import pandas as pd

def version_of(import_name):
    try:
        mod = __import__(import_name)
        return getattr(mod, '__version__', 'installed-version-unknown')
    except Exception as exc:
        return f'not installed ({exc.__class__.__name__})'

print('Python:', sys.version)
for name in ['torch', 'ultralytics', 'onnxruntime', 'cv2', 'numpy', 'pandas']:
    print(f'{name}:', version_of(name))

try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('nvidia-smi unavailable:', repr(exc))

def find_videos():
    videos = []
    for root in [KAGGLE_INPUT, KAGGLE_WORKING, Path.cwd()]:
        if root.exists():
            videos.extend([p.resolve() for p in root.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS])
    return sorted(set(videos))[:MAX_VIDEO_FILES]

VIDEO_FILES = find_videos()
print('Found video files:', len(VIDEO_FILES))
for p in VIDEO_FILES[:10]:
    print(p, p.stat().st_size)

## 3. Estimate Video Stream Size

When actual video files are present, file size and duration are used. Otherwise, continuous streaming size is estimated from bitrate and duration.

In [ ]:
def video_duration_seconds(path):
    try:
        import cv2
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 0
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
        cap.release()
        if fps > 0 and frames > 0:
            return frames / fps
    except Exception:
        pass
    return None

def continuous_size_from_bitrate(duration_s, mbps):
    return duration_s * mbps * 1_000_000 / 8.0

video_rows = []
if VIDEO_FILES:
    for path in VIDEO_FILES:
        duration = video_duration_seconds(path)
        size = path.stat().st_size
        video_rows.append({
            'scenario': path.name,
            'duration_seconds': duration,
            'continuous_video_bytes': size,
            'source': 'actual_file_size',
        })
else:
    video_rows.append({
        'scenario': 'assumed_1080p_stream',
        'duration_seconds': ASSUMED_DURATION_SECONDS,
        'continuous_video_bytes': continuous_size_from_bitrate(ASSUMED_DURATION_SECONDS, ASSUMED_BITRATE_MBPS),
        'source': f'assumed_{ASSUMED_BITRATE_MBPS}_mbps',
    })

video_df = pd.DataFrame(video_rows)
display(video_df)

## 4. Compare Cloud Payload Strategies

The edge pipeline sends only structured logs, or logs plus violation snapshots. This keeps raw video on the edge device and reduces bandwidth and privacy exposure.

In [ ]:
def alerts_for_duration(duration_s):
    if duration_s is None or duration_s <= 0:
        duration_s = ASSUMED_DURATION_SECONDS
    return max(1, int(round((duration_s / 3600.0) * ASSUMED_ALERTS_PER_HOUR)))

summary_rows = []
for row in video_df.to_dict('records'):
    duration = row['duration_seconds'] or ASSUMED_DURATION_SECONDS
    alerts = alerts_for_duration(duration)
    continuous = row['continuous_video_bytes']
    log_bytes = alerts * LOG_BYTES_PER_ALERT
    crop_bytes = log_bytes + alerts * SNAPSHOT_KB_PER_ALERT * 1024
    blur_bytes = log_bytes + alerts * BLURRED_SNAPSHOT_KB_PER_ALERT * 1024
    for strategy, sent_bytes in [
        ('json_csv_logs_only', log_bytes),
        ('logs_plus_violation_crops', crop_bytes),
        ('logs_plus_blurred_snapshots', blur_bytes),
    ]:
        reduction = 100.0 * (1.0 - sent_bytes / continuous) if continuous > 0 else None
        summary_rows.append({
            'scenario': row['scenario'],
            'source': row['source'],
            'duration_seconds': duration,
            'alerts_estimated': alerts,
            'cloud_strategy': strategy,
            'continuous_video_mb': continuous / (1024 * 1024),
            'edge_payload_mb': sent_bytes / (1024 * 1024),
            'bandwidth_reduction_percent': reduction,
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 5. Save Bandwidth Summary

The output table is written to `/kaggle/working/results/bandwidth_summary.csv`.

In [ ]:
out_csv = RESULTS_DIR / 'bandwidth_summary.csv'
out_json = RESULTS_DIR / 'bandwidth_summary.json'
summary_df.to_csv(out_csv, index=False)
out_json.write_text(summary_df.to_json(orient='records', indent=2) + '\n')
print('Saved:', out_csv)
print('Saved:', out_json)